# 05 - Model Evaluation & Performance Analysis
**Diabetic Retinopathy Stage Detection**

Rubric criterion 6 (15 marks): *accuracy, precision, recall, F1, confusion matrix, loss/accuracy
curves, insightful interpretation and error analysis.*

The **test set** (5,270 images, 15 % of patients, never used for any decision so far) is evaluated
exactly once here with the final model chosen in notebook 04.

Contents: 1. experiment recap - 2. test metrics - 3. confusion matrices - 4. training curves -
5. ROC / PR per class - 6. referable-DR screening view - 7. error analysis - 8. Grad-CAM - 9. summary.

In [ ]:
# --- Environment bootstrap: works on Kaggle, Ubuntu GPU box and Windows ----
import os, sys, subprocess, shutil
from pathlib import Path

REPO = "https://github.com/pradeesha999/dr-stage-detection-v2.git"
if Path("/kaggle/working").exists():
    dst = Path("/kaggle/working/dr_project")
    if not dst.exists():
        try:   # private repo: store a GitHub token as Kaggle secret GITHUB_TOKEN
            from kaggle_secrets import UserSecretsClient
            url = REPO.replace("https://", f"https://{UserSecretsClient().get_secret('GITHUB_TOKEN')}@")
        except Exception:
            url = REPO                      # works as-is if the repo is public
        subprocess.run(["git", "clone", "-q", url, str(dst)], check=True)
        subprocess.run(["git", "-C", str(dst), "remote", "set-url", "origin", REPO])  # keep token out of .git/config
    os.chdir(dst / "notebooks")
    # Trained models are git-ignored: pull them from a previous Kaggle run added as input.
    (dst / "outputs" / "models").mkdir(parents=True, exist_ok=True)
    for f in Path("/kaggle/input").rglob("outputs/models/*.keras"):
        if not (dst / "outputs" / "models" / f.name).exists():
            shutil.copy(f, dst / "outputs" / "models" / f.name); print("copied", f)
sys.path.insert(0, str(Path.cwd().parent))     # notebooks/ -> dr_project/
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import tensorflow as tf, keras
from sklearn.metrics import roc_curve, auc, precision_recall_curve

from src import config, data, utils, train
from src import evaluate as E
from src import preprocess as pp
utils.set_seed()
sns.set_theme(style="whitegrid")
print("TF", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))

train_df, val_df, test_df = data.load_splits()
model, cfg = E.load_model_fp32()
RUN = cfg["run"]
print(f"Final model: {RUN} | backbone={cfg['backbone']} | balancing={cfg['balancing']} | "
      f"val QWK={cfg['val_qwk']:.3f}")

## 1. Experiment recap (validation set, from notebook 04)

In [ ]:
tbl = train.results_table()
display(tbl)

## 2. Test-set metrics

In [ ]:
y_test = test_df.label.values
probs = E.predict_df(model, test_df)
y_pred = probs.argmax(1)
np.save(train.METRIC_DIR / "test_probs.npy", probs)
test_df.assign(pred=y_pred, **{f"p{i}": probs[:, i] for i in range(config.NUM_CLASSES)}) \
       .to_csv(train.METRIC_DIR / "test_predictions.csv", index=False)

overall = E.summary_metrics(y_test, probs)
(train.METRIC_DIR / "test_metrics.json").write_text(json.dumps(overall, indent=2))
display(pd.Series(overall, name="test").to_frame().round(4))

per_class = E.per_class_table(y_test, y_pred)
per_class.to_csv(train.METRIC_DIR / "test_per_class.csv")
display(per_class)

**How to read this.** *Accuracy* is inflated by the 73 % No_DR majority, so the more honest numbers
are **macro-F1** (every stage counts equally), **QWK** (rewards near-misses on the ordinal scale) and
**within-one-stage** (fraction of predictions at most one stage off - clinically often acceptable).
Per-class recall on Severe / Proliferate_DR is the number a clinician would ask for first: those are
the patients who must not be missed.

## 3. Confusion matrices

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
for a, norm, t in zip(ax, [False, True], ["counts", "row-normalised (recall)"]):
    cm = E.confusion(y_test, y_pred, normalize=norm)
    sns.heatmap(cm, annot=True, fmt=".2f" if norm else "d", cmap="Blues", ax=a,
                xticklabels=config.CLASS_NAMES, yticklabels=config.CLASS_NAMES, cbar=False)
    a.set(title=f"Confusion matrix - {t}", xlabel="predicted", ylabel="true")
    a.tick_params(axis="x", rotation=25)
plt.tight_layout(); utils.save_fig("05_confusion_matrix"); plt.show()

adjacent = np.abs(y_pred - y_test) == 1
print(f"errors: {(y_pred != y_test).mean():.1%} of test images | "
      f"of those, {adjacent.sum() / max((y_pred != y_test).sum(), 1):.0%} are off by exactly one stage")

## 4. Training curves of the final model (accuracy & loss)

In [ ]:
h = train.load_curves(RUN)
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].plot(h.accuracy, label="train"); ax[0].plot(h.val_accuracy, label="val"); ax[0].set_title("accuracy")
ax[1].plot(h.loss, label="train"); ax[1].plot(h.val_loss, label="val"); ax[1].set_title("loss")
ax[2].plot(h.val_qwk, label="val QWK"); ax[2].plot(h.val_macro_f1, label="val macro-F1"); ax[2].set_title("validation QWK / macro-F1")
for a in ax:
    a.axvline(cfg["epochs_head"] - 0.5, ls="--", c="gray", lw=1); a.set_xlabel("epoch"); a.legend()
plt.suptitle(f"{RUN}: dashed line = start of fine-tuning"); plt.tight_layout(); utils.save_fig("05_training_curves"); plt.show()

**Overfitting check.** Train and validation accuracy/loss should track each other; a widening gap
after the dashed line means the fine-tuned backbone starts memorising. Early stopping restores the
best-QWK epoch, so the model evaluated above is the peak of the validation curve, not the last epoch.

## 5. ROC and precision-recall curves (one-vs-rest)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
for i, name in enumerate(config.CLASS_NAMES):
    yi = (y_test == i).astype(int)
    fpr, tpr, _ = roc_curve(yi, probs[:, i]); ax[0].plot(fpr, tpr, label=f"{name} (AUC {auc(fpr, tpr):.3f})")
    pr, rc, _ = precision_recall_curve(yi, probs[:, i]); ax[1].plot(rc, pr, label=f"{name} (AP {auc(rc, pr):.3f})")
ax[0].plot([0, 1], [0, 1], "k--", lw=1); ax[0].set(title="ROC (one-vs-rest)", xlabel="false positive rate", ylabel="true positive rate"); ax[0].legend()
ax[1].set(title="Precision-Recall (one-vs-rest)", xlabel="recall", ylabel="precision"); ax[1].legend()
plt.tight_layout(); utils.save_fig("05_roc_pr_curves"); plt.show()

## 6. Referable-DR screening view
Clinically the key decision is binary: **refer** (Moderate NPDR or worse) vs **re-screen next year**.
Summing the probabilities of stages 2-4 gives a referral score; sweeping its threshold trades
sensitivity (catching disease) against specificity (not flooding the clinic).

In [ ]:
rows = [E.referable_metrics(y_test, probs, t) for t in np.linspace(0.1, 0.9, 17)]
ref = pd.DataFrame(rows)
display(ref[["threshold", "sensitivity", "specificity", "auc"]].round(3))

fig, ax = plt.subplots(figsize=(6, 4.2))
ax.plot(ref.threshold, ref.sensitivity, marker="o", label="sensitivity")
ax.plot(ref.threshold, ref.specificity, marker="s", label="specificity")
ax.set(xlabel="referral threshold on P(stage >= Moderate)", ylabel="", title=f"Referable DR - AUC {ref.auc.iloc[0]:.3f}")
ax.legend(); plt.tight_layout(); utils.save_fig("05_referable_dr"); plt.show()
ref.to_csv(train.METRIC_DIR / "test_referable.csv", index=False)

## 7. Error analysis

In [ ]:
# 7a. Where do errors come from? Distance between predicted and true stage.
dist = pd.Series(y_pred - y_test).value_counts().sort_index()
display(dist.rename("count").to_frame().T)

# 7b. Confidence of correct vs wrong predictions
conf = probs.max(1)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(conf[y_pred == y_test], bins=25, color="seagreen", label="correct", ax=ax[0], stat="density")
sns.histplot(conf[y_pred != y_test], bins=25, color="indianred", label="wrong", ax=ax[0], stat="density")
ax[0].set(title="Prediction confidence", xlabel="max softmax probability"); ax[0].legend()
sns.barplot(x=dist.index, y=dist.values, ax=ax[1], color="steelblue")
ax[1].set(title="Predicted - true stage", xlabel="stage difference", ylabel="count")
plt.tight_layout(); utils.save_fig("05_error_distribution"); plt.show()

# 7c. Worst mistakes: high-confidence, far from the truth
err = test_df.assign(pred=y_pred, conf=conf, gap=np.abs(y_pred - y_test))
worst = err[err.gap >= 2].sort_values("conf", ascending=False).head(8)
fig, axes = plt.subplots(2, 4, figsize=(15, 7.5)); axes = axes.ravel()
for a, (_, r) in zip(axes, worst.iterrows()):
    a.imshow(pp.read_rgb(r.proc_path)); a.axis("off")
    a.set_title(f"true {config.CLASS_NAMES[r.label]}\npred {config.CLASS_NAMES[r.pred]} ({r.conf:.2f})", fontsize=9)
plt.suptitle("Most confident errors that are >= 2 stages off"); plt.tight_layout(); utils.save_fig("05_worst_errors"); plt.show()

**Interpretation.** Most mistakes are between neighbouring stages (Mild <-> Moderate, Moderate <-> Severe)
where the grading criteria themselves are subtle (count of haemorrhages, presence of a few
microaneurysms). Large-gap errors are typically images with poor quality (blur, glare, dark
periphery) or with atypical lesions. Missing Severe / PDR (predicting a lower stage) is the
costliest error type and should be reduced by lowering the referral threshold, at the expense
of more false referrals - see section 6.

## 8. Grad-CAM: what does the network look at?
Heat should sit on lesions (haemorrhages, exudates, neovascularisation) - not on the optic disc,
image border or camera artefacts. This is the sanity check that the model learned pathology, not
shortcuts.

In [ ]:
cam = E.GradCAM(model)
fig, axes = plt.subplots(config.NUM_CLASSES, 4, figsize=(13, 3.3 * config.NUM_CLASSES))
for r, cname in enumerate(config.CLASS_NAMES):
    ok = err[(err.label == r) & (err.pred == r)].sort_values("conf", ascending=False).head(2)
    for c, (_, row) in enumerate(ok.iterrows()):
        img = pp.read_rgb(row.proc_path)
        heat, ci, p = cam(img.astype("float32"))
        axes[r, 2 * c].imshow(img); axes[r, 2 * c].set_title(f"{cname} - p={p:.2f}", fontsize=9)
        axes[r, 2 * c + 1].imshow(E.overlay(img, heat)); axes[r, 2 * c + 1].set_title("Grad-CAM", fontsize=9)
    for a in axes[r]: a.axis("off")
plt.tight_layout(); utils.save_fig("05_gradcam_per_class"); plt.show()

# Grad-CAM on a few errors: what fooled it?
fig, axes = plt.subplots(1, 6, figsize=(16, 3.2))
for i, (_, row) in enumerate(worst.head(3).iterrows()):
    img = pp.read_rgb(row.proc_path); heat, ci, p = cam(img.astype("float32"))
    axes[2 * i].imshow(img); axes[2 * i].set_title(f"true {config.CLASS_NAMES[row.label]}", fontsize=9)
    axes[2 * i + 1].imshow(E.overlay(img, heat)); axes[2 * i + 1].set_title(f"pred {config.CLASS_NAMES[row.pred]} ({p:.2f})", fontsize=9)
for a in axes: a.axis("off")
plt.tight_layout(); utils.save_fig("05_gradcam_errors"); plt.show()

## 9. Summary table for the report

In [ ]:
summary = {
    "final model": f"{cfg['backbone']} ({cfg['balancing']}, img {cfg['img_size']}, {cfg['epochs_run']} epochs)",
    "test accuracy": overall["accuracy"], "test macro-F1": overall["macro_f1"],
    "test weighted-F1": overall["weighted_f1"], "test QWK": overall["qwk"],
    "test macro AUC (OvR)": overall["auc_ovr_macro"], "within one stage": overall["within_one_stage"],
    "referable-DR AUC": ref.auc.iloc[0],
}
s = pd.Series(summary, name="value"); display(s.to_frame())

**Figures produced:** confusion matrices, training curves, ROC/PR, referable-DR trade-off,
error distribution, worst errors, Grad-CAM per class and on errors - all in `outputs/figures/05_*.png`.
**Files:** `test_metrics.json`, `test_per_class.csv`, `test_predictions.csv`, `test_referable.csv`.